# LP vs MILP vs LP+Price Floor

This notebook compares three dispatch formulations for handling simultaneous
charge/discharge in the η < 1 case:

| Approach | scenario_id | Mechanism |
|---|---|---|
| LP (baseline) | `actual__lp_dr__eta090__deg000` | No constraint; c and d may both be > 0 |  # noqa: E501
| MILP (correct) | `actual__milp_dr__eta090__deg000` | Binary z[t] enforces mutual exclusivity |  # noqa: E501
| LP + price floor | `actual__lp_floor_dr__eta090__deg000` | Clip prices to 0 before LP |  # noqa: E501

Revenue is always settled at actual prices. The LP "cheat" — simultaneously
charging and discharging at negative-price hours — is only worth a fraction of a
percent, so the LP is a safe approximation for revenue forecasting.

## 1. Setup

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from vpp.paths import ProjPaths

paths = ProjPaths()
paths.ensure_directories()

plt.rcParams["figure.dpi"] = 100
sns.set_theme(style="whitegrid")

CAPACITY_KWH = 100.0
ETA_RT = 0.90
_MIN_ACTIVE = 0.01  # kW — below this is numerical noise

## 2. Load Data

In [2]:
prices_raw = pd.read_parquet(paths.smard_prices_file)
prices_eur_mwh = prices_raw["price_de_lu"].dropna().sort_index()
prices_berlin = prices_eur_mwh.copy()
prices_berlin.index = prices_berlin.index.tz_convert("Europe/Berlin")

_SCENARIO_IDS_3 = [
    "actual__lp_dr__eta090__deg000",
    "actual__milp_dr__eta090__deg000",
    "actual__lp_floor_dr__eta090__deg000",
]
_SCENARIO_LABEL = {
    "actual__lp_dr__eta090__deg000": "LP (baseline)",
    "actual__milp_dr__eta090__deg000": "MILP (correct)",
    "actual__lp_floor_dr__eta090__deg000": "LP + price floor",
}

dispatch_raw = pd.read_parquet(
    paths.dispatch_schedules_file,
    filters=[("scenario_id", "in", _SCENARIO_IDS_3)],
)

prices_df = (
    prices_berlin.rename("price_eur_mwh")
    .reset_index()
    .rename(columns={"index": "timestamp"})
)
dispatch_merged = dispatch_raw.merge(prices_df, on="timestamp")
dispatch_merged["revenue_eur"] = (
    dispatch_merged["price_eur_mwh"]
    * (dispatch_merged["d"] - dispatch_merged["c"])
    / 1000
)
dispatch_merged["simul"] = (dispatch_merged["c"] > _MIN_ACTIVE) & (
    dispatch_merged["d"] > _MIN_ACTIVE
)
dispatch_merged["neg_price"] = dispatch_merged["price_eur_mwh"] < 0
dispatch_merged["date"] = dispatch_merged["timestamp"].dt.date
dispatch_merged["year"] = dispatch_merged["timestamp"].dt.year

n_days = dispatch_merged[
    dispatch_merged["scenario_id"] == "actual__lp_dr__eta090__deg000"
]["date"].nunique()
ann = 365.25 / n_days

# Annual revenue per scenario
rev_by_scenario = (
    dispatch_merged.groupby("scenario_id")["revenue_eur"].sum() * ann
).to_dict()

lp_rev = rev_by_scenario["actual__lp_dr__eta090__deg000"]
milp_rev = rev_by_scenario["actual__milp_dr__eta090__deg000"]
floor_rev = rev_by_scenario["actual__lp_floor_dr__eta090__deg000"]

milp_gap = (milp_rev / lp_rev - 1) * 100
floor_gap = (floor_rev / lp_rev - 1) * 100

print(f"Annual revenue (eta_rt={ETA_RT}, daily reset, 100 kWh / 50 kW):")
print(f"  LP (baseline):       {lp_rev:,.0f} EUR/yr")
print(f"  MILP (correct):      {milp_rev:,.0f} EUR/yr")
print(f"  LP + price floor:    {floor_rev:,.0f} EUR/yr")
print(f"  MILP vs LP gap:      {milp_gap:+.3f}%")
print(f"  Price floor vs LP:   {floor_gap:+.2f}%")

# Simultaneous C+D stats (LP only)
lp_dispatch = dispatch_merged[
    dispatch_merged["scenario_id"] == "actual__lp_dr__eta090__deg000"
]
simul_mask = lp_dispatch["simul"]
neg_and_simul = lp_dispatch["neg_price"] & simul_mask
n_total = len(lp_dispatch)
n_simul = simul_mask.sum()
n_neg_simul = neg_and_simul.sum()

print("\nLP simultaneous C+D:")
print(f"  Total: {n_simul:,} / {n_total:,} hours ({n_simul / n_total * 100:.2f}%)")
print(
    f"  At negative prices: {n_neg_simul:,} "
    f"({n_neg_simul / max(n_simul, 1) * 100:.0f}% of simultaneous)"
)

Annual revenue (eta_rt=0.9, daily reset, 100 kWh / 50 kW):
  LP (baseline):       3,490 EUR/yr
  MILP (correct):      3,481 EUR/yr
  LP + price floor:    3,401 EUR/yr
  MILP vs LP gap:      -0.266%
  Price floor vs LP:   -2.54%

LP simultaneous C+D:
  Total: 1,390 / 67,896 hours (2.05%)
  At negative prices: 1,390 (100% of simultaneous)


## 3. Revenue and Simultaneous C+D by Year

In [3]:
yearly_stats = (
    dispatch_merged.groupby(["scenario_id", "year"])
    .agg(
        annual_rev=(
            "revenue_eur",
            lambda x: (
                x.sum()
                * 365.25
                / dispatch_merged.loc[
                    dispatch_merged["scenario_id"] == x.name[0], "date"
                ].nunique()
            ),
        ),
        n_hours=("simul", "count"),
        simul_hours=("simul", "sum"),
    )
    .reset_index()
)
# Simpler: compute annual revenue from daily aggregation
daily_rev = (
    dispatch_merged.groupby(["scenario_id", "year", "date"])["revenue_eur"]
    .sum()
    .reset_index()
)
yearly_rev = (
    daily_rev.groupby(["scenario_id", "year"])["revenue_eur"]
    .agg(lambda x: x.sum() * 365.25 / len(x))
    .reset_index()
    .rename(columns={"revenue_eur": "annual_rev_eur"})
)

lp_yearly = dispatch_merged[
    dispatch_merged["scenario_id"] == "actual__lp_dr__eta090__deg000"
]
simul_by_year = (
    lp_yearly.groupby("year")
    .agg(
        simul_pct=("simul", lambda x: x.mean() * 100),
        neg_simul_pct=(
            "simul",
            lambda x: (x & lp_yearly.loc[x.index, "neg_price"]).mean() * 100,
        ),
    )
    .reset_index()
)

print("\nAnnual revenue by scenario and year:")
yearly_pivot = yearly_rev.pivot(
    index="year", columns="scenario_id", values="annual_rev_eur"
)
yearly_pivot.columns = [_SCENARIO_LABEL.get(c, c) for c in yearly_pivot.columns]
print(yearly_pivot.round(0).to_string())


Annual revenue by scenario and year:
      LP (baseline)  LP + price floor  MILP (correct)
year                                                 
2018         1280.0            1276.0          1280.0
2019         1057.0            1012.0          1048.0
2020         1161.0            1094.0          1151.0
2021         2834.0            2798.0          2830.0
2022         6867.0            6863.0          6867.0
2023         3580.0            3480.0          3572.0
2024         4107.0            3992.0          4095.0
2025         4594.0            4438.0          4581.0
2026         5099.0            4764.0          5067.0


/tmp/ipykernel_1456574/4104934366.py:7: RuntimeWarning: divide by zero encountered in scalar divide
  x.sum()


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: annual revenue by scenario (bar chart)
approaches = ["LP\n(baseline)", "MILP\n(correct)", "LP +\nprice floor"]
revenues = [lp_rev, milp_rev, floor_rev]
colors = ["#1f77b4", "#2ca02c", "#ff7f0e"]

bars = axes[0].bar(approaches, revenues, color=colors, edgecolor="white", width=0.5)
axes[0].set_ylabel("Annualised revenue (EUR/year)")
axes[0].set_title("Revenue comparison by approach")
ymax = max(revenues) * 1.18
axes[0].set_ylim(0, ymax)
for bar, rev in zip(bars, revenues):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + ymax * 0.01,
        f"{rev:,.0f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )
axes[0].text(
    0.5,
    0.92,
    f"MILP vs LP: {milp_gap:+.3f}%\nFloor vs LP: {floor_gap:+.2f}%",
    ha="center",
    transform=axes[0].transAxes,
    fontsize=8.5,
    bbox={"boxstyle": "round,pad=0.3", "facecolor": "lightyellow", "edgecolor": "gray"},
)

# Right: simultaneous C+D by year (LP only)
x = np.arange(len(simul_by_year))
axes[1].bar(
    x,
    simul_by_year["simul_pct"],
    color="#1f77b4",
    alpha=0.8,
    label="All simultaneous",
)
axes[1].bar(
    x,
    simul_by_year["neg_simul_pct"],
    color="#d62728",
    alpha=0.8,
    label="Simul. at neg. price",
)
axes[1].set_xticks(x)
axes[1].set_xticklabels(simul_by_year["year"])
axes[1].set_ylabel("% of all hours")
axes[1].set_title("Simultaneous C+D hours by year (LP)")
axes[1].legend(fontsize=8)

fig.suptitle(
    f"LP vs MILP vs LP+floor (eta_rt={ETA_RT}, daily reset, 100 kWh / 50 kW)",
    fontsize=11,
)
fig.tight_layout()
fig.savefig(paths.images_path / "11_simultaneous_cd.png", dpi=150, bbox_inches="tight")
plt.show()

/tmp/ipykernel_1456574/3991555752.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


```{figure} ../../output/images/11_simultaneous_cd.png
:name: fig-11-simultaneous-cd
Left: annualised revenue for each approach; the LP–MILP gap (the value of the
physically-infeasible simultaneous dispatch) is annotated as a percentage.
Right: fraction of hours with simultaneous C+D in the LP by year, split by
negative-price vs positive-price hours.
```

## 4. Summary

Key findings replicated from this cross-scenario view:

- The LP simultaneous C+D "cheat" is worth only ~0.27% of annual revenue.
  The LP is therefore a safe proxy for revenue forecasting, even though it
  produces physically invalid dispatch schedules in ~2% of hours.
- All simultaneous-dispatch hours in the LP occur at negative-price hours,
  confirming the mechanism: the LP "burns" SoC through efficiency losses to
  create headroom for additional gross charging when prices are negative.
- The price-floor fix is counterproductive: it forfeits genuine negative-price
  charging revenue (worth ~2.5% of total), far exceeding the LP cheat it removes.
- MILP is the correct and practical solution. With daily-reset sub-problems of
  24 binary variables, CBC solves it as fast as the LP relaxation.